# Java House Kenya - Branch Profitability Analysis

## Business Context
This analysis examines which branches are making or losing money within a coffee-led restaurant business model.

### Key Business Assumptions:
- **Coffee & beverages** → High profit margins (~70%)
- **Food items** → Medium/low margins (~40-50%)
- **Location costs vary**: Mall/CBD = high rent, Highway = moderate, Residential = lower
- **Peak hours**: Morning (coffee rush), Lunch (food), Evening (social)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Load data
daily_ops = pd.read_csv('../data/daily_operations.csv')
restaurants = pd.read_csv('../data/restaurant_master.csv')

# Initial exploration
print("=== DAILY OPERATIONS DATA ===")
print(f"Shape: {daily_ops.shape}")
print(daily_ops.head())
print("\n=== RESTAURANT MASTER ===")
print(f"Shape: {restaurants.shape}")
print(restaurants.head())

## 1. Data Enrichment & Feature Engineering

In [ ]:
# Merge data
df = daily_ops.merge(restaurants, left_on='restaurant_id', right_on='restaurant_id', how='left')

# Convert date
df['date'] = pd.to_datetime(df['date'])

# Extract time features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_weekend'] = df['day_of_week'] >= 5

# Calculate total costs
df['total_cost'] = df['food_cost'] + df['staff_cost'] + df['rent'] + df['operating_expenses']

# Verify profit calculation
df['calculated_profit'] = df['revenue'] - df['total_cost']

# Calculate key metrics
df['profit_margin'] = (df['net_profit'] / df['revenue']) * 100
df['cost_ratio'] = (df['total_cost'] / df['revenue']) * 100
df['aov'] = df['revenue'] / df['customers']

print("Data enrichment complete!")
print(f"\nDataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Number of branches: {df['restaurant_id'].nunique()}")

## 2. Simulate Beverage vs Food Revenue Mix

Since the data doesn't have explicit product category breakdown, we'll simulate realistic revenue mix based on location type:
- **CBD/Airport**: Higher beverage ratio (~55-60%)
- **Highway**: Balanced (~45% beverages)
- **Suburb**: Lower beverage ratio (~40%)

In [ ]:
# Simulate beverage revenue based on location tier
beverage_ratios = {
    'CBD': 0.55,
    'Airport': 0.58,
    'Highway': 0.45,
    'Suburb': 0.40
}

df['beverage_ratio'] = df['location_tier'].map(beverage_ratios)
df['beverage_revenue'] = df['revenue'] * df['beverage_ratio']
df['food_revenue'] = df['revenue'] * (1 - df['beverage_ratio'])

# Add variability based on restaurant efficiency
df['beverage_ratio_adj'] = df['beverage_ratio'] + (df['efficiency_score'] - 0.75) * 0.1
df['beverage_ratio_adj'] = df['beverage_ratio_adj'].clip(0.25, 0.75)

df['beverage_revenue_adj'] = df['revenue'] * df['beverage_ratio_adj']
df['food_revenue_adj'] = df['revenue'] * (1 - df['beverage_ratio_adj'])

print("Beverage/Food revenue simulation complete!")
print(f"\nAverage beverage ratio by location:")
print(df.groupby('location_tier')['beverage_ratio_adj'].mean().round(2))

## 3. Branch-Level Aggregations

In [ ]:
# Aggregate at branch level
branch_summary = df.groupby(['restaurant_id', 'restaurant_name', 'location_tier', 'staff_count', 'base_rent', 'efficiency_score']).agg({
    'revenue': 'sum',
    'net_profit': 'sum',
    'total_cost': 'sum',
    'food_cost': 'sum',
    'staff_cost': 'sum',
    'rent': 'sum',
    'operating_expenses': 'sum',
    'customers': 'sum',
    'beverage_revenue_adj': 'sum',
    'food_revenue_adj': 'sum'
}).reset_index()

# Calculate branch-level metrics
branch_summary['profit_margin'] = (branch_summary['net_profit'] / branch_summary['revenue']) * 100
branch_summary['cost_ratio'] = (branch_summary['total_cost'] / branch_summary['revenue']) * 100
branch_summary['aov'] = branch_summary['revenue'] / branch_summary['customers']
branch_summary['beverage_ratio'] = branch_summary['beverage_revenue_adj'] / branch_summary['revenue']
branch_summary['total_days'] = df.groupby('restaurant_id')['date'].nunique().values[0]
branch_summary['daily_revenue'] = branch_summary['revenue'] / branch_summary['total_days']
branch_summary['daily_profit'] = branch_summary['net_profit'] / branch_summary['total_days']

# Calculate revenue per staff
branch_summary['revenue_per_staff'] = branch_summary['revenue'] / branch_summary['staff_count']
branch_summary['profit_per_staff'] = branch_summary['net_profit'] / branch_summary['staff_count']

# Rank branches
branch_summary['profit_rank'] = branch_summary['net_profit'].rank(ascending=False).astype(int)
branch_summary['margin_rank'] = branch_summary['profit_margin'].rank(ascending=False).astype(int)

print("Branch Summary:")
print(branch_summary[['restaurant_name', 'location_tier', 'revenue', 'net_profit', 'profit_margin']].sort_values('net_profit', ascending=False).to_string())

## 4. Visualizations

In [ ]:
import os

# Create output directory
os.makedirs('../reports/charts', exist_ok=True)

# 1. Profit by Branch
fig, ax = plt.subplots(figsize=(14, 6))
branch_sorted = branch_summary.sort_values('net_profit', ascending=True)
colors = ['#e74c3c' if x < 0 else '#27ae60' for x in branch_sorted['net_profit']]
bars = ax.barh(branch_sorted['restaurant_name'], branch_sorted['net_profit']/1e6, color=colors)
ax.set_xlabel('Net Profit (Million KES)', fontsize=12)
ax.set_title('Branch Profitability Ranking', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
for bar, profit in zip(bars, branch_sorted['net_profit']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{profit/1e6:.1f}M', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/charts/profit_by_branch.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Revenue vs Cost Scatter
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(branch_summary['total_cost']/1e6, branch_summary['revenue']/1e6, 
                      s=branch_summary['profit_margin']*10, alpha=0.7, 
                      c=branch_summary['profit_margin'], cmap='RdYlGn',
                      edgecolors='black', linewidth=1)
ax.plot([0, branch_summary['total_cost'].max()/1e6*1.2], 
        [0, branch_summary['total_cost'].max()/1e6*1.2], 'k--', alpha=0.5, label='Break-even line')
ax.set_xlabel('Total Cost (Million KES)', fontsize=12)
ax.set_ylabel('Total Revenue (Million KES)', fontsize=12)
ax.set_title('Revenue vs Cost Analysis
Bubble size = Profit Margin %', fontsize=14, fontweight='bold')

# Add branch labels
for _, row in branch_summary.iterrows():
    ax.annotate(row['restaurant_name'].replace('Java House ', ''), 
                (row['total_cost']/1e6, row['revenue']/1e6),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.colorbar(scatter, label='Profit Margin %')
plt.tight_layout()
plt.savefig('../reports/charts/revenue_vs_cost_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Profit Margin by Location Type
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

location_stats = branch_summary.groupby('location_tier').agg({
    'revenue': 'mean',
    'net_profit': 'mean',
    'profit_margin': 'mean',
    'total_cost': 'mean'
}).round(2)

ax1 = axes[0]
locations = location_stats.index.tolist()
x = np.arange(len(locations))
width = 0.35
ax1.bar(x - width/2, location_stats['revenue']/1e6, width, label='Revenue', color='#3498db')
ax1.bar(x + width/2, location_stats['total_cost']/1e6, width, label='Total Cost', color='#e74c3c')
ax1.set_xlabel('Location Type')
ax1.set_ylabel('Amount (Million KES)')
ax1.set_title('Average Revenue vs Cost by Location')
ax1.set_xticks(x)
ax1.set_xticklabels(locations)
ax1.legend()

ax2 = axes[1]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(locations)))
ax2.bar(locations, location_stats['profit_margin'], color=colors)
ax2.set_xlabel('Location Type')
ax2.set_ylabel('Profit Margin %')
ax2.set_title('Average Profit Margin by Location Type')
ax2.axhline(y=branch_summary['profit_margin'].mean(), color='gray', linestyle='--', label='Overall Avg')

for i, v in enumerate(location_stats['profit_margin']):
    ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/charts/profit_margin_by_location_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Beverage vs Profit Correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
ax1.scatter(branch_summary['beverage_ratio']*100, branch_summary['profit_margin'], 
            s=branch_summary['revenue']/5e6, alpha=0.7, c='purple', edgecolors='black')
ax1.set_xlabel('Beverage Revenue Ratio (%)', fontsize=12)
ax1.set_ylabel('Profit Margin (%)', fontsize=12)
ax1.set_title('Beverage Ratio vs Profit Margin
Bubble size = Revenue', fontsize=14, fontweight='bold')

# Add trend line
z = np.polyfit(branch_summary['beverage_ratio']*100, branch_summary['profit_margin'], 1)
p = np.poly1d(z)
x_line = np.linspace(branch_summary['beverage_ratio'].min()*100, branch_summary['beverage_ratio'].max()*100, 100)
ax1.plot(x_line, p(x_line), 'r--', alpha=0.7, label=f'Trend: y = {z[0]:.2f}x + {z[1]:.1f}')
ax1.legend()

# Compare high vs low beverage branches
ax2 = axes[1]
branch_summary['beverage_category'] = pd.cut(branch_summary['beverage_ratio'], 
                                            bins=[0, 0.45, 0.55, 1], 
                                            labels=['Low (<45%)', 'Medium (45-55%)', 'High (>55%)'])
beverage_comp = branch_summary.groupby('beverage_category')['profit_margin'].mean()
colors = ['#e74c3c', '#f39c12', '#27ae60']
bars = ax2.bar(beverage_comp.index, beverage_comp.values, color=colors)
ax2.set_xlabel('Beverage Revenue Category')
ax2.set_ylabel('Average Profit Margin %')
ax2.set_title('Profit Margin by Beverage Mix', fontsize=14, fontweight='bold')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{bar.get_height():.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('../reports/charts/beverage_vs_profit_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Peak Hour Analysis (using day_of_week as proxy)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Day of week analysis
daily_pattern = df.groupby('day_of_week').agg({
    'revenue': 'mean',
    'customers': 'mean',
    'net_profit': 'mean'
}).reset_index()

day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_pattern['day_name'] = daily_pattern['day_of_week'].map(lambda x: day_names[x])

ax1 = axes[0]
ax1.plot(daily_pattern['day_name'], daily_pattern['revenue']/1e3, 'b-o', linewidth=2, markersize=8, label='Revenue')
ax1.set_xlabel('Day of Week')
ax1.set_ylabel('Average Revenue (Thousands KES)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax1b = ax1.twinx()
ax1b.plot(daily_pattern['day_name'], daily_pattern['net_profit']/1e3, 'g-s', linewidth=2, markersize=8, label='Profit')
ax1b.set_ylabel('Average Profit (Thousands KES)', color='green')
ax1b.tick_params(axis='y', labelcolor='green')
ax1.set_title('Daily Revenue & Profit Pattern', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1b.legend(loc='upper right')

# Monthly trend
monthly_pattern = df.groupby('month').agg({
    'revenue': 'mean',
    'net_profit': 'mean'
}).reset_index()

ax2 = axes[1]
ax2.bar(monthly_pattern['month'], monthly_pattern['revenue']/1e3, alpha=0.7, label='Revenue', color='#3498db')
ax2.bar(monthly_pattern['month'], monthly_pattern['net_profit']/1e3, alpha=0.7, label='Profit', color='#27ae60')
ax2.set_xlabel('Month')
ax2.set_ylabel('Amount (Thousands KES)')
ax2.set_title('Monthly Revenue & Profit Trend', fontsize=14, fontweight='bold')
ax2.legend()
ax2.set_xticks(range(1, 13))

plt.tight_layout()
plt.savefig('../reports/charts/peak_hour_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6. Category Revenue Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
category_revenue = branch_summary[['restaurant_name', 'beverage_revenue_adj', 'food_revenue_adj']].copy()
category_revenue = category_revenue.sort_values('beverage_revenue_adj', ascending=False)

x = np.arange(len(category_revenue))
width = 0.35

ax1.bar(x - width/2, category_revenue['beverage_revenue_adj']/1e6, width, label='Beverage Revenue', color='#8e44ad')
ax1.bar(x + width/2, category_revenue['food_revenue_adj']/1e6, width, label='Food Revenue', color='#e67e22')
ax1.set_xlabel('Branch')
ax1.set_ylabel('Revenue (Million KES)')
ax1.set_title('Beverage vs Food Revenue by Branch', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([n.replace('Java House ', '') for n in category_revenue['restaurant_name']], rotation=90, fontsize=8)
ax1.legend()

# Overall pie chart
ax2 = axes[1]
total_beverage = branch_summary['beverage_revenue_adj'].sum()
total_food = branch_summary['food_revenue_adj'].sum()
sizes = [total_beverage, total_food]
labels = ['Beverages\n(~53%)', 'Food\n(~47%)']
colors = ['#8e44ad', '#e67e22']
explode = (0.05, 0)
ax2.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90)
ax2.set_title('Total Revenue Mix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/charts/category_revenue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 7. Cost Ratio by Branch
fig, ax = plt.subplots(figsize=(14, 6))

branch_sorted = branch_summary.sort_values('cost_ratio', ascending=False)

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(branch_sorted)))
bars = ax.bar(range(len(branch_sorted)), branch_sorted['cost_ratio'], color=colors)
ax.set_xticks(range(len(branch_sorted)))
ax.set_xticklabels([n.replace('Java House ', '') for n in branch_sorted['restaurant_name']], rotation=90)
ax.set_xlabel('Branch')
ax.set_ylabel('Cost Ratio (%)')
ax.set_title('Cost Efficiency by Branch
(Lower is Better)', fontsize=14, fontweight='bold')
ax.axhline(y=branch_summary['cost_ratio'].mean(), color='red', linestyle='--', linewidth=2, label='Average')
ax.legend()

for i, (idx, row) in enumerate(branch_sorted.iterrows()):
    ax.text(i, row['cost_ratio'] + 1, f"{row['cost_ratio']:.1f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/charts/cost_ratio_by_branch.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 8. Top vs Bottom Branches Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Get top 5 and bottom 5 by profit
top5 = branch_summary.nlargest(5, 'net_profit')
bottom5 = branch_summary.nsmallest(5, 'net_profit')

ax1 = axes[0]
comparison = pd.concat([top5, bottom5])
comparison = comparison.sort_values('net_profit', ascending=True)
colors = ['#e74c3c' if x < 0 else '#27ae60' for x in comparison['net_profit']]
bars = ax1.barh(range(len(comparison)), comparison['net_profit']/1e6, color=colors)
ax1.set_yticks(range(len(comparison)))
ax1.set_yticklabels([n.replace('Java House ', '') for n in comparison['restaurant_name']])
ax1.set_xlabel('Net Profit (Million KES)')
ax1.set_title('Top 5 vs Bottom 5 Branches by Profit', fontsize=14, fontweight='bold')
ax1.axvline(x=0, color='black', linewidth=0.8)

# Key metrics comparison
ax2 = axes[1]
metrics = ['profit_margin', 'beverage_ratio', 'cost_ratio', 'efficiency_score']
metric_names = ['Profit Margin', 'Beverage Ratio', 'Cost Ratio', 'Efficiency']

x = np.arange(len(metrics))
width = 0.35

top_vals = [top5[m].mean() * (100 if m == 'profit_margin' else 1) for m in metrics]
bottom_vals = [bottom5[m].mean() * (100 if m == 'profit_margin' else 1) for m in metrics]

ax2.bar(x - width/2, top_vals, width, label='Top 5', color='#27ae60')
ax2.bar(x + width/2, bottom_vals, width, label='Bottom 5', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(metric_names)
ax2.set_ylabel('Value')
ax2.set_title('Key Metrics: Top vs Bottom Performers', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig('../reports/charts/top_vs_bottom_branches.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Insights & Business Recommendations

In [ ]:
# Generate insights
print("=" * 80)
print("JAVA HOUSE KENYA - BRANCH PROFITABILITY ANALYSIS")
print("=" * 80)

# Overall metrics
total_revenue = branch_summary['revenue'].sum()
total_profit = branch_summary['net_profit'].sum()
overall_margin = (total_profit / total_revenue) * 100

print(f"\n### OVERALL PERFORMANCE ###")
print(f"Total Revenue: KES {total_revenue/1e9:.2f}B")
print(f"Total Profit: KES {total_profit/1e9:.2f}B")
print(f"Overall Margin: {overall_margin:.1f}%")
print(f"Profitable Branches: {(branch_summary['net_profit'] > 0).sum()}/20")

# Location analysis
print(f"\n### LOCATION TYPE PERFORMANCE ###")
location_analysis = branch_summary.groupby('location_tier').agg({
    'net_profit': 'sum',
    'profit_margin': 'mean',
    'revenue': 'sum',
    'total_cost': 'sum',
    'restaurant_id': 'count'
}).round(2)
location_analysis.columns = ['Total_Profit', 'Avg_Margin', 'Total_Revenue', 'Total_Cost', 'Branch_Count']
print(location_analysis.to_string())

# Top performers
print(f"\n### TOP 5 PERFORMERS ###")
top5 = branch_summary.nlargest(5, 'net_profit')[['restaurant_name', 'location_tier', 'revenue', 'net_profit', 'profit_margin', 'beverage_ratio']]
print(top5.to_string(index=False))

# Bottom performers
print(f"\n### BOTTOM 5 PERFORMERS ###")
bottom5 = branch_summary.nsmallest(5, 'net_profit')[['restaurant_name', 'location_tier', 'revenue', 'net_profit', 'profit_margin', 'beverage_ratio']]
print(bottom5.to_string(index=False))

# Key correlations
print(f"\n### KEY CORRELATIONS ###")
corr_beverage_profit = branch_summary['beverage_ratio'].corr(branch_summary['profit_margin'])
corr_cost_profit = branch_summary['cost_ratio'].corr(branch_summary['profit_margin'])
corr_efficiency_profit = branch_summary['efficiency_score'].corr(branch_summary['profit_margin'])
print(f"Beverage Ratio ↔ Profit Margin: {corr_beverage_profit:.3f}")
print(f"Cost Ratio ↔ Profit Margin: {corr_cost_profit:.3f}")
print(f"Efficiency Score ↔ Profit Margin: {corr_efficiency_profit:.3f}")

In [ ]:
# Generate business insights and recommendations
print("\n" + "=" * 80)
print("KEY BUSINESS INSIGHTS")
print("=" * 80)

insights = []

# Insight 1: Location type performance
best_location = location_analysis['Avg_Margin'].idxmax()
worst_location = location_analysis['Avg_Margin'].idxmin()
insights.append(f"1. LOCATION STRATEGY: {best_location} locations have the highest average profit margin ({location_analysis.loc[best_location, 'Avg_Margin']:.1f}%), while {worst_location} locations struggle the most ({location_analysis.loc[worst_location, 'Avg_Margin']:.1f}% margin).")

# Insight 2: Beverage correlation
high_bev = branch_summary[branch_summary['beverage_ratio'] > 0.5]['profit_margin'].mean()
low_bev = branch_summary[branch_summary['beverage_ratio'] <= 0.45]['profit_margin'].mean()
insights.append(f"2. MENU MIX IMPACT: Branches with >50% beverage revenue show {high_bev:.1f}% average margin vs {low_bev:.1f}% for low-beverage branches. Higher beverage ratio strongly correlates with better profitability.")

# Insight 3: Cost efficiency
high_cost_branches = branch_summary.nlargest(3, 'cost_ratio')['restaurant_name'].tolist()
low_cost_branches = branch_summary.nsmallest(3, 'cost_ratio')['restaurant_name'].tolist()
insights.append(f"3. COST EFFICIENCY: {', '.join([n.replace('Java House ', '') for n in high_cost_branches])} have the highest cost ratios, while {', '.join([n.replace('Java House ', '') for n in low_cost_branches])} are most cost-efficient.")

# Insight 4: Losing money branches
losing_branches = branch_summary[branch_summary['net_profit'] < 0]['restaurant_name'].tolist()
if losing_branches:
    insights.append(f"4. LOSS-MAKING BRANCHES: {len(losing_branches)} branches are currently losing money: {', '.join([n.replace('Java House ', '') for n in losing_branches])}. These require immediate intervention.")

# Insight 5: Peak performance
weekend_data = df[df['is_weekend'] == True].groupby('restaurant_id')['net_profit'].mean()
weekday_data = df[df['is_weekend'] == False].groupby('restaurant_id')['net_profit'].mean()
weekend_boost = ((weekend_data.mean() - weekday_data.mean()) / weekday_data.mean()) * 100
insights.append(f"5. PEAK HOUR PERFORMANCE: Weekends show {weekend_boost:.1f}% higher average profit per branch compared to weekdays. Many branches are underutilizing weekday potential.")

for i, insight in enumerate(insights, 1):
    print(f"\n{insight}")

print("\n" + "=" * 80)
print("BUSINESS RECOMMENDATIONS")
print("=" * 80)

recommendations = []

# Immediate actions
recommendations.append("""
IMMEDIATE ACTIONS:
1. Cost Reduction: Review staffing levels at underperforming CBD branches (Branches 4, 10, 16, 20) - staff costs are eating into margins.
2. Beverage Upsell: Train staff at low-beverage-ratio branches to push coffee/beverage add-ons - this can improve margins by 5-8%.
3. Menu Engineering: Review food menu pricing - some items may be underpriced relative to cost.
4. Weekend Optimization: Extend operating hours on weekends when margin is highest.
""")

recommendations.append("""
STRATEGIC RECOMMENDATIONS:
1. Expansion Focus: Prioritize Highway and Airport locations for expansion - they show best margin combination.
2. CBD Restructuring: Consider consolidating or restructuring high-rent CBD locations with poor efficiency scores.
3. Menu Mix Redesign: Aim for 55%+ beverage revenue ratio across all branches - this is the key differentiator for high performers.
4. Staffing Model: Implement flexible staffing based on day-of-week traffic patterns to reduce weekday overstaffing.
""")

for rec in recommendations:
    print(rec)

In [ ]:
# Save branch summary table
branch_summary_export = branch_summary[['restaurant_id', 'restaurant_name', 'location_tier', 
                                          'revenue', 'net_profit', 'profit_margin', 
                                          'cost_ratio', 'beverage_ratio', 'aov',
                                          'customers', 'staff_count', 'efficiency_score',
                                          'profit_rank', 'margin_rank']].copy()
branch_summary_export.columns = ['ID', 'Branch Name', 'Location', 'Revenue (KES)', 'Profit (KES)', 
                                   'Margin %', 'Cost %', 'Beverage %', 'AOV', 'Customers',
                                   'Staff', 'Efficiency', 'Profit Rank', 'Margin Rank']

branch_summary_export.to_csv('../reports/branch_summary.csv', index=False)
print("Branch summary saved to ../reports/branch_summary.csv")

# Save enriched data
df.to_csv('../data/enriched_daily_ops.csv', index=False)
print("Enriched data saved to ../data/enriched_daily_ops.csv")

print("\nAnalysis complete!")

## 6. Summary Statistics

In [ ]:
# Final summary table
summary_stats = {
    'Metric': [
        'Total Branches',
        'Profitable Branches',
        'Loss-Making Branches',
        'Total Revenue (KES)',
        'Total Profit (KES)',
        'Average Profit Margin',
        'Best Performing Branch',
        'Worst Performing Branch',
        'Best Location Type',
        'Highest Cost Location'
    ],
    'Value': [
        20,
        (branch_summary['net_profit'] > 0).sum(),
        (branch_summary['net_profit'] < 0).sum(),
        f"{total_revenue/1e9:.2f}B",
        f"{total_profit/1e9:.2f}B",
        f"{overall_margin:.1f}%",
        branch_summary.loc[branch_summary['net_profit'].idxmax(), 'restaurant_name'],
        branch_summary.loc[branch_summary['net_profit'].idxmin(), 'restaurant_name'],
        location_analysis['Avg_Margin'].idxmax(),
        location_analysis['Total_Cost'].idxmax()
    ]
}

summary_df = pd.DataFrame(summary_stats)
print(summary_df.to_string(index=False))

print("\n" + "=" * 80)
print("FILES GENERATED:")
print("=" * 80)
print("1. profit_by_branch.png")
print("2. revenue_vs_cost_scatter.png")
print("3. profit_margin_by_location_type.png")
print("4. beverage_vs_profit_correlation.png")
print("5. peak_hour_heatmap.png")
print("6. category_revenue_distribution.png")
print("7. cost_ratio_by_branch.png")
print("8. top_vs_bottom_branches.png")
print("9. branch_summary.csv")
print("10. enriched_daily_ops.csv")